# Entrenamiento del Modelo de IA para Diagnóstico de Hojas de Uvillas
**Autor:** Marlon Esteban Yajamin Remache

## 1. Configuración del Entorno y Estructuración de Datos (15%)
En esta fase preparamos el entorno de Google Colab instalando las dependencias críticas (`ultralytics` para el framework YOLO y `roboflow` para la ingesta de datos). El dataset de las hojas de uvillas está particionado estratégicamente (Train/Valid/Test) directamente desde el workspace, garantizando que el entorno `.yaml` mantenga la configuración de las 4 clases de severidad requeridas.

In [3]:
# Instalación de librerías base
!pip install ultralytics roboflow

# Importaciones necesarias
from ultralytics import YOLO
from roboflow import Roboflow
import yaml

# Descarga del dataset de hojas de uvillas (Workspace: estebans-workspace-vnyrh)
rf = Roboflow(api_key="aVN16Xexix6U5V1EsJVq")
project = rf.workspace("estebans-workspace-vnyrh").project("hojas-de-ubillas-vapwm")
dataset = project.version(2).download("yolov11")

# Verificación de la estructura de datos
ruta_yaml = f"{dataset.location}/data.yaml"
with open(ruta_yaml, 'r') as file:
    config_yaml = file.read()
    print("\n--- Estructura del data.yaml ---")
    print(config_yaml)

loading Roboflow workspace...
loading Roboflow project...

--- Estructura del data.yaml ---
train: ../train/images
val: ../valid/images
test: ../test/images

nc: 3
names: ['Grave', 'Leve', 'Medio']

roboflow:
  workspace: estebans-workspace-vnyrh
  project: hojas-de-ubillas-vapwm
  version: 2
  license: Private
  url: https://app.roboflow.com/estebans-workspace-vnyrh/hojas-de-ubillas-vapwm/2


## 2. Entrenamiento y Afinación de Hiperparámetros (20%)
**Aplicación de Transfer Learning y Prevención de Sobreajuste (Overfitting):**
Para este entrenamiento, partimos de los pesos base del modelo `yolo26n.pt` (Transfer Learning). Aunque configuramos `epochs=200`, activamos el hiperparámetro `patience=25` (Early Stopping) para que la red detenga el aprendizaje automáticamente si la pérdida de validación deja de mejorar; esto evita que el modelo memorice las hojas en lugar de aprender sus patrones.

Adicionalmente, se aplican técnicas de aumento de datos (*Data Augmentation*) nativas como el volteo horizontal (`fliplr=0.5`) y variaciones de saturación HSV (`hsv_s=0.2`) para robustecer la visión artificial ante cambios de iluminación en el cultivo real.

In [ ]:
# Inicialización del modelo base (Transfer Learning)
model = YOLO("yolo26n.pt")

# Entrenamiento con afinación de hiperparámetros contra el sobreajuste
results = model.train(
    data=ruta_yaml,
    epochs=150,
    patience=25,         # Early Stopping: Detiene si no mejora en 25 épocas
    imgsz=640,
    batch=16,
    device=0,            # Utilización de hardware acelerado T4 GPU
    fliplr=0.5,          # Aumento de datos: Volteo horizontal aleatorio
    hsv_s=0.2            # Aumento de datos: Alteración de saturación
)

Ultralytics 8.4.103 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Hojas-De-Ubillas-2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.2, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, ov

## 3. Meta de Precisión Algorítmica: Evaluación mAP (40%)
El siguiente gráfico consolida el rendimiento del entrenamiento. El objetivo crítico de este proyecto es alcanzar un **mAP@0.5 igual o superior al 82%** para el conjunto completo de las 4 clases fenológicas evaluadas (Sana, Leve, Medio, Grave). La curva de validación confirma el hito de precisión de la red neuronal.

In [ ]:
from IPython.display import Image, display

# Mostramos las gráficas de resultados generadas por YOLO
display(Image(filename=f'{results.save_dir}/results.png', width=1000))

## 4. Documentación Profesional: Análisis de la Matriz de Confusión (10%)
La Matriz de Confusión normalizada nos permite auditar cómo la IA clasifica el nivel de infección foliar.

Al observar la diagonal principal, se evidencia una fuerte retención de Verdaderos Positivos en todas las clases. Las confusiones marginales (Falsos Negativos/Positivos) tienden a presentarse únicamente entre clases adyacentes (por ejemplo, clasificar una mancha 'Leve' como 'Medio'). Esto es un comportamiento biológicamente esperado debido a que la progresión del hongo en la hoja es un espectro continuo. Lo más importante es que existe una nula confusión entre los extremos (clasificar 'Sana' como 'Grave'), lo cual asegura que el sistema de control automatizado en campo no disparará una orden de fumigación severa por error.

In [ ]:
# Mostramos la Matriz de Confusión Normalizada
display(Image(filename=f'{results.save_dir}/confusion_matrix_normalized.png', width=800))

## 5. Inferencia Final en Simulación (15%)
Validación del lazo de control. En esta celda se extraen imágenes del conjunto de prueba (*Test dataset*) —imágenes que la red neuronal nunca ha visto durante sus 200 épocas de entrenamiento— para simular el comportamiento que tendrá el modelo exportado a NCNN al ejecutarse en la Raspberry Pi 5.

In [ ]:
import glob

# Inferencia simulada sobre el conjunto de test
predicciones = model.predict(
    source=f"{dataset.location}/test/images",
    conf=0.5,
    save=True
)

# Extraer y visualizar 3 ejemplos procesados del lote de prueba
directorio_prediccion = predicciones[0].save_dir
imagenes_procesadas = glob.glob(f'{directorio_prediccion}/*.jpg')[:3]

print("\n--- Resultados de Detección Simulada en el Borde ---")
for img_path in imagenes_procesadas:
    display(Image(filename=img_path, width=400))